# Reflection Removal — Evaluation Pipeline

This notebook evaluates all trained checkpoints and visualises results.

**What it does:**
1. Loads every checkpoint (Baseline → Stage 1 → Final)
2. Computes PSNR and SSIM on the real test set (SIR2)
3. Prints a comparison table
4. Shows side-by-side visual results


## 0. Install Dependencies

In [ ]:
!pip install -q torchvision scikit-image pytorch-msssim


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Imports & Device

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3. Configure Paths

In [ ]:
REAL_PATH  = "/content/drive/MyDrive/Reflection_Removal/datasets/SIR2/training_pairs"
MODEL_DIR  = "/content/drive/MyDrive/Reflection_Removal/models"

# Checkpoints to compare (name → filename)
MODEL_PATHS = {
    "Baseline"  : os.path.join(MODEL_DIR, "baseline.pth"),
    "Stage1"    : os.path.join(MODEL_DIR, "improved_model_stage1.pth"),
    "Final_V3"  : os.path.join(MODEL_DIR, "final_model_v3.pth"),
}


## 4. Real Dataset & Loader

In [ ]:
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])


class RealDataset(Dataset):
    """SIR2 real dataset — A/ (blended), B/ (ground-truth)."""

    def __init__(self, root_dir):
        self.input_dir = os.path.join(root_dir, "A")
        self.gt_dir    = os.path.join(root_dir, "B")
        self.files     = sorted(os.listdir(self.input_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f   = self.files[idx]
        inp = Image.open(os.path.join(self.input_dir, f)).convert("RGB")
        gt  = Image.open(os.path.join(self.gt_dir,    f)).convert("RGB")
        return transform(inp), transform(gt)


real_dataset = RealDataset(REAL_PATH)
real_loader  = DataLoader(real_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Test samples: {len(real_dataset)}")


## 5. Model Definitions

In [ ]:
class RefractionModule(nn.Module):
    """Predicts a 2-D optical flow and spatially warps the input image."""

    def __init__(self):
        super().__init__()
        self.flow_predictor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32,  2, 3, padding=1),
        )

    def forward(self, x):
        flow = self.flow_predictor(x)
        B, C, H, W = x.shape

        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, H, device=x.device),
            torch.linspace(-1, 1, W, device=x.device),
            indexing='ij'
        )
        grid = torch.stack((grid_x, grid_y), dim=2).unsqueeze(0).expand(B, -1, -1, -1)

        flow = flow.permute(0, 2, 3, 1)
        flow = flow / torch.tensor([W, H], dtype=torch.float32, device=x.device)

        warped = F.grid_sample(x, grid + flow, align_corners=True)
        return warped, flow


class ResNetUNet(nn.Module):
    """ResNet-50 encoder + transposed-conv decoder. Outputs are in [0, 1]."""

    def __init__(self):
        super().__init__()
        base   = models.resnet50(weights="IMAGENET1K_V1")
        layers = list(base.children())

        self.layer0 = nn.Sequential(*layers[:3])
        self.layer1 = nn.Sequential(*layers[3:5])
        self.layer2 = layers[5]
        self.layer3 = layers[6]
        self.layer4 = layers[7]

        self.up4   = nn.ConvTranspose2d(2048, 1024, 2, 2)
        self.up3   = nn.ConvTranspose2d(1024,  512, 2, 2)
        self.up2   = nn.ConvTranspose2d( 512,  256, 2, 2)
        self.up1   = nn.ConvTranspose2d( 256,   64, 2, 2)
        self.final = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        out = self.final(self.up1(self.up2(self.up3(self.up4(x4)))))
        out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        return torch.sigmoid(out)


class ReflectionRemovalModel(nn.Module):
    """Full pipeline: RefractionModule → ResNetUNet."""

    def __init__(self):
        super().__init__()
        self.refraction = RefractionModule()
        self.unet       = ResNetUNet()

    def forward(self, x):
        aligned, _ = self.refraction(x)
        out        = self.unet(aligned)
        return out, aligned


## 6. Checkpoint Loader

In [ ]:
def load_model(name, path):
    """
    Instantiate the correct model class for each checkpoint and load
    weights, skipping any layers whose shapes don't match (safe loading).
    """
    print(f"Loading {name} from '{path}'...")

    model = ResNetUNet().to(device) if name == "Baseline" else ReflectionRemovalModel().to(device)

    checkpoint  = torch.load(path, map_location=device)
    model_state = model.state_dict()

    compatible = {
        k: v for k, v in checkpoint.items()
        if k in model_state and v.shape == model_state[k].shape
    }
    model_state.update(compatible)
    model.load_state_dict(model_state)

    print(f"  Loaded {len(compatible)}/{len(model_state)} layers.")
    model.eval()
    return model


## 7. Evaluation Function

In [ ]:
MAX_BATCHES = 100   # limit for speed; set to None to evaluate all


def evaluate(model, loader, is_baseline=False):
    """
    Returns mean PSNR and SSIM over the dataset.
    is_baseline=True uses ResNetUNet (single output);
    is_baseline=False uses ReflectionRemovalModel (pred, aligned).
    """
    psnr_scores, ssim_scores = [], []

    with torch.no_grad():
        for i, (inp, gt) in enumerate(loader):
            if MAX_BATCHES is not None and i >= MAX_BATCHES:
                break

            inp, gt = inp.to(device), gt.to(device)

            pred = model(inp) if is_baseline else model(inp)[0]
            pred = torch.clamp(pred, 0, 1)

            pred_np = pred.cpu().numpy().transpose(0, 2, 3, 1)
            gt_np   = gt.cpu().numpy().transpose(0, 2, 3, 1)

            for p, g in zip(pred_np, gt_np):
                psnr_scores.append(psnr_fn(g, p, data_range=1.0))
                ssim_scores.append(ssim_fn(g, p, channel_axis=2, data_range=1.0))

    return float(np.mean(psnr_scores)), float(np.mean(ssim_scores))


## 8. Run Evaluation — All Checkpoints

In [ ]:
results = {}

for name, path in MODEL_PATHS.items():
    model = load_model(name, path)
    is_baseline = (name == "Baseline")
    p, s = evaluate(model, real_loader, is_baseline)
    results[name] = (p, s)
    print(f"  {name:12s} → PSNR: {p:.3f} dB | SSIM: {s:.4f}")
    print()


## 9. Comparison Table

In [ ]:
print("=" * 45)
print(f"{'Model':<14} {'PSNR (dB)':>10} {'SSIM':>10}")
print("-" * 45)
for name, (p, s) in results.items():
    print(f"{name:<14} {p:>10.3f} {s:>10.4f}")
print("=" * 45)


## 10. Visual Comparison

In [ ]:
# Load the best (final) model for visualisation
final_model = load_model("Final_V3", MODEL_PATHS["Final_V3"])
final_model.eval()

NUM_IMAGES = 4   # number of examples to display

indices = random.sample(range(len(real_dataset)), NUM_IMAGES)

fig, axes = plt.subplots(NUM_IMAGES, 3, figsize=(12, 4 * NUM_IMAGES))
fig.suptitle("Reflection Removal Results (Final Model)", fontsize=14, fontweight="bold")

column_titles = ["Input (blended)", "Ground Truth", "Model Output"]
for ax, title in zip(axes[0], column_titles):
    ax.set_title(title, fontsize=11)

for row, idx in enumerate(indices):
    inp, gt = real_dataset[idx]

    with torch.no_grad():
        pred, _ = final_model(inp.unsqueeze(0).to(device))

    inp_np  = inp.numpy().transpose(1, 2, 0)
    gt_np   = gt.numpy().transpose(1, 2, 0)
    pred_np = pred.squeeze().cpu().numpy().transpose(1, 2, 0)
    pred_np = np.clip(pred_np, 0, 1)

    for col, img in enumerate([inp_np, gt_np, pred_np]):
        axes[row][col].imshow(img)
        axes[row][col].axis("off")

plt.tight_layout()
plt.show()


## Evaluation Complete

The table above shows how each training stage improves PSNR and SSIM on the
real SIR2 test set. The visual grid gives a qualitative sense of reflection
removal quality across random samples.
